这份代码是**推荐系统的「排序阶段」完整实现**，承接前面的**召回环节**，核心是给用户的候选文章精准打分、排序，最终输出每人Top5推荐文章。

### 一、核心定位
- 召回：**粗筛**，从海量文章里快速筛出几百个候选，缩小范围。
- 排序：**精排**，对候选集精准计算点击概率，按概率从高到低排序，选出最贴合用户的5篇。

### 二、主要做了这4件事
1. **读取并预处理排序特征**
    加载用户、文章、用户-文章交互的特征（相似度、时间差、点击历史、类别等），清洗数据、规整格式。
2. **训练3种经典排序模型打分**
    - **LGB排序模型**：专门做排序任务，优化推荐排序指标（NDCG）。
    - **LGB分类模型**：把“是否点击”转为二分类，输出点击概率。
    - **DIN深度模型**：阿里的深度兴趣网络，捕捉用户历史行为与候选文章的相关性，更精准刻画用户兴趣。
3. **5折交叉验证**
    按用户划分折数，防止数据泄露，输出模型预测分数/排名，用于后续融合。
4. **模型融合提升效果**
    - 加权融合：直接累加3个模型的预测分数，综合排序。
    - Stacking：用逻辑回归拟合3个模型的输出，二次预测更稳。

### 三、最终产出
生成**每个用户Top5推荐文章**的CSV提交文件，是推荐系统的最终输出结果。

简单说：你接下来要做的，就是**用特征训练排序模型→融合模型→输出最终推荐列表**。

In [1]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import gc, os
import time
from datetime import datetime
import lightgbm as lgb
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
data_path = './data_raw/'
save_path = './temp_results/'
offline = False

In [3]:
# 重新读取数据的时候，发现click_article_id是一个浮点数，所以将其转换成int类型
trn_user_item_feats_df = pd.read_csv(save_path + 'trn_user_item_feats_df.csv')
trn_user_item_feats_df['click_article_id'] = trn_user_item_feats_df['click_article_id'].astype(int)

if offline:
    val_user_item_feats_df = pd.read_csv(save_path + 'val_user_item_feats_df.csv')
    val_user_item_feats_df['click_article_id'] = val_user_item_feats_df['click_article_id'].astype(int)
else:
    val_user_item_feats_df = None
    
tst_user_item_feats_df = pd.read_csv(save_path + 'tst_user_item_feats_df.csv')
tst_user_item_feats_df['click_article_id'] = tst_user_item_feats_df['click_article_id'].astype(int)

# 做特征的时候为了方便，给测试集也打上了一个无效的标签，这里直接删掉就行
del tst_user_item_feats_df['label']

### 本部分是推荐系统**排序阶段收尾**的两个核心工具函数，分工明确：
1. submit()：将模型预测的排序结果，按用户提取Top5推荐文章，转换成标准提交格式并保存CSV文件；
2. norm_sim()：对预测分数做**最小-最大归一化**，把分数缩放到0~1区间，消除不同模型分数量级差异，为模型融合做准备。

### norm_sim() 函数的使用场景与核心作用
一句话总结：**这个函数是为「模型融合」服务的**，解决不同模型输出分数「量级不统一、无法直接相加/融合」的问题。     

推荐系统里我们训练了 3 个不同的排序模型，它们输出的 pred_score（预测分数）范围完全不一样：         
LGB 分类模型：输出是 0~1 之间的概率值（比如 0.7 表示 70% 点击概率）；         
LGB 排序模型：输出是「相对排序分」，范围可能是 -5~5 或者 0~100（没有固定上下限）；    
DIN 深度模型：输出同样是 0~1 概率，但分布可能和 LGB 分类模型不一样。    
如果直接把这 3 个分数加起来做融合，会出现一个问题：               
比如 LGB 排序模型的分数是 0~100，另外两个是 0~1，直接相加的话，LGB 排序模型的分数会「主导」结果，另外两个模型的影响几乎被淹没，融合就没意义了。          
norm_sim () 的作用：把所有模型的分数统一缩放到 0~1 之间，让每个模型的「话语权」平等，融合才有效。      

场景 1：加权融合前（直接对预测分数归一化）       
看代码最后的「模型融合 - 加权融合」部分：          
场景 2：五折交叉验证后（对 Stacking 特征归一化）      
看 LGB 排序模型的五折交叉验证部分：

In [4]:
# 生成标准提交结果函数
def submit(recall_df, topk=5, model_name=None):
    # 按用户ID分组，对预测分数升序排序（为后续倒序排名做准备）
    recall_df = recall_df.sort_values(by=['user_id', 'pred_score'])
    # 按用户分组，对预测分数降序排名，method='first'保证相同分数不并列
    recall_df['rank'] = recall_df.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')
    
    # 校验：确保每个用户至少有topk篇候选文章，不满足直接报错
    tmp = recall_df.groupby('user_id').apply(lambda x: x['rank'].max())
    assert tmp.min() >= topk
    
    # 删除预测分数列，仅保留ID和排名
    del recall_df['pred_score']
    # 筛选每个用户Topk的文章，行转列（宽表格式）适配提交规范
    submit = recall_df[recall_df['rank'] <= topk].set_index(['user_id', 'rank']).unstack(-1).reset_index()
    
    # 格式化列名，清理多层索引
    submit.columns = [int(col) if isinstance(col, int) else col for col in submit.columns.droplevel(0)]
    # 按照标准提交格式重命名列
    submit = submit.rename(columns={'': 'user_id', 1: 'article_1', 2: 'article_2', 
                                                  3: 'article_3', 4: 'article_4', 5: 'article_5'})
    
    # 拼接保存路径+文件名，导出CSV文件
    save_name = save_path + model_name + '_' + datetime.today().strftime('%m-%d') + '.csv'
    submit.to_csv(save_name, index=False, header=True)

# 预测分数归一化函数（用于模型融合）
def norm_sim(sim_df, weight=0.0):
    # 计算分数的最小值和最大值
    min_sim = sim_df.min()
    max_sim = sim_df.max()
    # 特殊情况：所有分数相同，直接赋值为1.0
    if max_sim == min_sim:
        sim_df = sim_df.apply(lambda sim: 1.0)
    # 常规情况：最小-最大归一化，将分数缩放到0~1
    else:
        sim_df = sim_df.apply(lambda sim: 1.0 * (sim - min_sim) / (max_sim - min_sim))

    # 给归一化后的分数加偏移权重（可选）
    sim_df = sim_df.apply(lambda sim: sim + weight)
    return sim_df

# LGB

## 一、LGB（LightGBM）是什么？
一句话：**一个非常好用的「机器学习模型」，专门用来做预测/排序，速度快、效果好**。
- 类比：就像一个「超级做题家」，你给它一堆题目（特征）和答案（标签），它能快速学会规律，然后做新题（预测）。
- 优势：比传统模型快几十倍，能处理大量数据，推荐系统里的「标配模型」。

---

## 二、LGB排序模型 vs LGB分类模型，分别干啥？
推荐系统里，我们的目标是「给用户推荐最可能点击的文章」，有两种实现思路：

### 1. LGB分类模型（Classification）
- **思路**：把问题变成「判断题」——这篇文章用户「会点击」还是「不会点击」？
- **输出**：0~1之间的概率值（比如 0.8 表示 80% 概率点击）。
- **类比**：老师给试卷打分，0分（不点击）到100分（点击），分数越高越推荐。

### 2. LGB排序模型（Ranking）
- **思路**：把问题变成「排序题」——给用户的几篇候选文章，按「点击可能性」从高到低排个序。
- **输出**：相对分数（没有固定范围，只看相对大小）。
- **类比**：运动会跑步比赛，不看具体跑了多少秒，只看谁是第一、谁是第二，按名次推荐。
- **核心区别**：排序模型更关注「相对顺序」，这正是推荐系统最需要的（我们只关心哪篇文章排第一，不关心具体分数是多少）。

---

## 三、单模型训练 vs 五折交叉验证，分别干啥？
### 1. 单模型训练
- **思路**：用全部训练数据训练一个模型，直接预测测试集。
- **作用**：快速跑通流程，看模型效果，生成一个基础的推荐结果。
- **类比**：平时做一套练习题，直接上考场。

### 2. 五折交叉验证
- **思路**：把训练数据分成5份，每次用4份训练、1份验证，重复5次，最后把5次的结果平均。
- **作用**：
  1. 更准确地评估模型效果（避免一次运气好/不好）；
  2. 生成「模型融合」需要的新特征（Stacking用）。
- **类比**：平时做5套练习题，每次模拟考试，最后取平均成绩，更稳。

---

## 四、你接下来会看到的代码流程
1. **单模型训练**：先训练一个LGB排序模型，生成推荐结果；
2. **五折交叉验证**：用更严谨的方式再训练一次LGB排序模型，生成融合用的特征；
3. **LGB分类模型**：换一种思路（分类），再训练一个模型；
4. **DIN深度模型**：用深度学习再训练一个模型；
5. **模型融合**：把3个模型的结果结合起来，得到最终的推荐。

## **LGB排序模型的单模型训练**

In [5]:
# 核心流程：数据备份→定义特征→按用户分组→定义模型→训练→预测→保存结果→生成提交文件

# 1. 数据备份：防止修改原始数据，重新读取麻烦
trn_user_item_feats_df_rank_model = trn_user_item_feats_df.copy()
if offline:
    val_user_item_feats_df_rank_model = val_user_item_feats_df.copy()
tst_user_item_feats_df_rank_model = tst_user_item_feats_df.copy()

# 2. 定义特征列：告诉模型用哪些数据来预测
lgb_cols = ['sim0', 'time_diff0', 'word_diff0','sim_max', 'sim_min', 'sim_sum', 
            'sim_mean', 'score','click_size', 'time_diff_mean', 'active_level',
            'click_environment','click_deviceGroup', 'click_os', 'click_country', 
            'click_region','click_referrer_type', 'user_time_hob1', 'user_time_hob2',
            'words_hbo', 'category_id', 'created_at_ts','words_count']

# 3. 排序模型分组：按用户分组，这是LGB排序模型的特殊要求
# 按用户 ID 排序，并统计每个用户有多少篇候选文章
# LGB 排序模型是「按用户」学习的，它需要知道哪些文章属于同一个用户，才能给这个用户的文章排顺序
trn_user_item_feats_df_rank_model.sort_values(by=['user_id'], inplace=True)
g_train = trn_user_item_feats_df_rank_model.groupby(['user_id'], as_index=False).count()["label"].values
if offline:
    val_user_item_feats_df_rank_model.sort_values(by=['user_id'], inplace=True)
    g_val = val_user_item_feats_df_rank_model.groupby(['user_id'], as_index=False).count()["label"].values

# 4. 定义LGB排序模型：设置模型参数
lgb_ranker = lgb.LGBMRanker(boosting_type='gbdt', num_leaves=31, reg_alpha=0.0, reg_lambda=1,
                            max_depth=-1, n_estimators=100, subsample=0.7, colsample_bytree=0.7, subsample_freq=1,
                            learning_rate=0.01, min_child_weight=50, random_state=2018, n_jobs=16)  
        # boosting_type='gbdt'：用梯度提升树（最常用的类型）；
        # num_leaves=31：每棵树的叶子节点数（控制模型复杂度）；
        # learning_rate=0.01：学习率（越小越稳，但训练越慢）；
        # n_estimators=100：树的数量（越多越准，但可能过拟合）；
        # reg_alpha=0.0, reg_lambda=1：正则化参数（防止过拟合）；
        # random_state=2018：随机种子（保证每次运行结果一样）；
        # n_jobs=16：用 16 个 CPU 核心并行计算（速度快）。

# 5. 训练模型：分离线调试和正式训练两种模式
if offline:
    # 离线模式 用ndcg指标评估 验证集效果50轮没提升则提前停止
    lgb_ranker.fit(trn_user_item_feats_df_rank_model[lgb_cols], trn_user_item_feats_df_rank_model['label'], group=g_train,
                eval_set=[(val_user_item_feats_df_rank_model[lgb_cols], val_user_item_feats_df_rank_model['label'])], 
                eval_group=[g_val], eval_at=[1, 2, 3, 4, 5], eval_metric=['ndcg'], early_stopping_rounds=50)
else:
    # 正式模式 无验证集，全部训练数据训练
    lgb_ranker.fit(trn_user_item_feats_df[lgb_cols], trn_user_item_feats_df['label'], group=g_train)

# 6. 模型预测：对测试集打分
# num_iteration=lgb_ranker.best_iteration_：用效果最好的那一轮树来预测（离线模式下有用，正式模式下用全部树）
tst_user_item_feats_df['pred_score'] = lgb_ranker.predict(tst_user_item_feats_df[lgb_cols], num_iteration=lgb_ranker.best_iteration_)

# 7. 保存预测结果：留着后面模型融合用
tst_user_item_feats_df[['user_id', 'click_article_id', 'pred_score']].to_csv(save_path + 'lgb_ranker_score.csv', index=False)

# 8. 生成提交结果：调用前面的 submit 函数，生成Top5推荐
rank_results = tst_user_item_feats_df[['user_id', 'click_article_id', 'pred_score']]
rank_results['click_article_id'] = rank_results['click_article_id'].astype(int)
submit(rank_results, topk=5, model_name='lgb_ranker')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005287 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4129
[LightGBM] [Info] Number of data points in the train set: 291135, number of used features: 23


以上【仅用 LGB 排序模型这一个模型】**，给测试集里的所有用户，单独生成的**初步推荐文章列表**，输出成了标准格式的 CSV 文件。
⚠️ 重点：这只是**单模型的临时结果**，**不是整个项目的最终推荐结果**！

# 五折交叉验证
在看代码前，先把「五折交叉验证在这里的核心作用」和「与前面单模型的区别」讲明白，这是**模型融合（Stacking）的关键前置步骤**。

---

## 一、这部分和前面「单模型训练」的核心区别
### 前面的单模型训练：
- **思路**：用全部训练数据训练1个模型，直接预测测试集；
- **问题**：
  1. 只用1次验证，效果评估可能不准（运气成分）；
  2. 无法生成「Stacking融合」需要的新特征。

### 这里的五折交叉验证：
- **思路**：把训练数据**按用户分成5份**，每次用4份训练、1份验证，重复5次；
- **核心目的**：
  1. **生成Stacking用的新特征**：给训练集和测试集都加上「LGB排序模型的预测分数和排名」；
  2. **更稳的评估**：5次结果平均，避免单次运气好坏。

---

## 二、为什么要「以用户为目标划分五折」？
这是推荐系统的**关键细节**，防止「数据泄露」：
- ❌ 错误做法：随机划分，可能把同一个用户的文章分到训练集和验证集；
- ✅ 正确做法：按用户划分，保证同一个用户的所有文章要么全在训练集，要么全在验证集；
- 类比：考试时，不能让学生在练习题里见过考试题，按用户划分就是保证「验证集的用户，训练集里完全没见过」。

---

## 三、这部分最终产出什么？
不是最终推荐结果，而是**2个给后面模型融合用的特征文件**：
1. `trn_lgb_ranker_feats.csv`：训练集的新特征（包含LGB排序模型的预测分数和排名）；
2. `tst_lgb_ranker_feats.csv`：测试集的新特征（同上）；
这两个文件是后面「Stacking模型融合」的核心素材！

In [6]:
# 定义按用户划分五折的函数
def get_kfold_users(trn_df, n=5):
    # 保证同一个用户的所有文章要么全在训练集，要么全在验证集，防止数据泄露
    # 获取所有唯一的用户ID
    user_ids = trn_df['user_id'].unique()
    # 按步长n切片，把用户分成5份（比如用户1,6,11...在第1份，2,7,12...在第2份）
    user_set = [user_ids[i::n] for i in range(n)]
    return user_set

# 初始化变量
k_fold = 5  # 五折交叉验证
trn_df = trn_user_item_feats_df_rank_model  # 用前面备份的训练集数据
user_set = get_kfold_users(trn_df, n=k_fold)  # 按用户划分好的5份用户列表

score_list = []  # 用来存每一轮验证集的预测结果 最后拼接成训练集的新特征
score_df = trn_df[['user_id', 'click_article_id', 'label']]  # 训练集的基础信息（用户、文章、标签） 后面和预测结果合并
sub_preds = np.zeros(tst_user_item_feats_df_rank_model.shape[0])  # 初始化测试集的预测分数数组（全0） 后面累加 5 次预测结果

# 五折交叉验证循环（核心部分）
for n_fold, valid_user in enumerate(user_set):
    # 1 划分训练集和验证集：验证集是当前这一折的用户，训练集是剩下的4折用户
    train_idx = trn_df[~trn_df['user_id'].isin(valid_user)]  # ~表示“非”，即不在验证集用户里的所有数据作为训练集
    valid_idx = trn_df[trn_df['user_id'].isin(valid_user)]
    
    # 2 按用户分组（LGB排序模型的特殊要求）
    train_idx.sort_values(by=['user_id'], inplace=True)
    g_train = train_idx.groupby(['user_id'], as_index=False).count()["label"].values
    
    valid_idx.sort_values(by=['user_id'], inplace=True)
    g_val = valid_idx.groupby(['user_id'], as_index=False).count()["label"].values
    
    # 3 定义LGB排序模型（和前面单模型参数一样）
    lgb_ranker = lgb.LGBMRanker(boosting_type='gbdt', num_leaves=31, reg_alpha=0.0, reg_lambda=1,
                            max_depth=-1, n_estimators=100, subsample=0.7, colsample_bytree=0.7, subsample_freq=1,
                            learning_rate=0.01, min_child_weight=50, random_state=2018, n_jobs=16)
    
    # 4 训练模型（用当前折的训练集，验证集评估） 早停防止过拟合
    lgb_ranker.fit(train_idx[lgb_cols], train_idx['label'], group=g_train,
                   eval_set=[(valid_idx[lgb_cols], valid_idx['label'])], eval_group=[g_val],
                   eval_at=[1, 2, 3, 4, 5], eval_metric=['ndcg'], callbacks=[lgb.early_stopping(stopping_rounds=50)])
    
    # 5 预测验证集结果
    valid_idx['pred_score'] = lgb_ranker.predict(valid_idx[lgb_cols], num_iteration=lgb_ranker.best_iteration_)
    
    # 6 对验证集预测分数归一化（为后面模型融合做准备）
    valid_idx['pred_score'] = valid_idx[['pred_score']].transform(lambda x: norm_sim(x))
    
    # 7 给验证集的文章按用户排名
    valid_idx.sort_values(by=['user_id', 'pred_score'])
    valid_idx['pred_rank'] = valid_idx.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')
    
    # 8 把验证集的预测结果存到列表里（后面拼接成训练集的新特征）
    score_list.append(valid_idx[['user_id', 'click_article_id', 'pred_score', 'pred_rank']])
    
    # 9 预测测试集结果（如果是正式模式，把5次预测分数累加到sub_preds里，最后求平均）
    if not offline:
        sub_preds += lgb_ranker.predict(tst_user_item_feats_df_rank_model[lgb_cols], lgb_ranker.best_iteration_)

# 拼接训练集的新特征并保存
score_df_ = pd.concat(score_list, axis=0)  # 把5折的验证集结果拼起来（覆盖整个训练集）
score_df = score_df.merge(score_df_, how='left', on=['user_id', 'click_article_id'])  # 和训练集基础信息合并
# 保存训练集的新特征（给后面Stacking用）  trn_lgb_ranker_feats.csv是给后面 Stacking 模型融合用的，为核心产出
score_df[['user_id', 'click_article_id', 'pred_score', 'pred_rank', 'label']].to_csv(save_path + 'trn_lgb_ranker_feats.csv', index=False)

# 平均测试集的预测结果，生成测试集的新特征并保存
tst_user_item_feats_df_rank_model['pred_score'] = sub_preds / k_fold  # 5次预测分数求平均
tst_user_item_feats_df_rank_model['pred_score'] = tst_user_item_feats_df_rank_model['pred_score'].transform(lambda x: norm_sim(x))  # 归一化
tst_user_item_feats_df_rank_model.sort_values(by=['user_id', 'pred_score'])
tst_user_item_feats_df_rank_model['pred_rank'] = tst_user_item_feats_df_rank_model.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')  # 排名
# 保存测试集的新特征（给后面Stacking用）
tst_user_item_feats_df_rank_model[['user_id', 'click_article_id', 'pred_score', 'pred_rank']].to_csv(save_path + 'tst_lgb_ranker_feats.csv', index=False)

# 生成单模型提交结果（可选，主要是看五折交叉验证后的单模型效果）
# 但核心目的还是前面生成的两个特征文件，给后面模型融合用！
rank_results = tst_user_item_feats_df_rank_model[['user_id', 'click_article_id', 'pred_score']]
rank_results['click_article_id'] = rank_results['click_article_id'].astype(int)
submit(rank_results, topk=5, model_name='lgb_ranker')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001439 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4128
[LightGBM] [Info] Number of data points in the train set: 233171, number of used features: 23
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[99]	valid_0's ndcg@1: 0.948325	valid_0's ndcg@2: 0.978515	valid_0's ndcg@3: 0.980115	valid_0's ndcg@4: 0.980287	valid_0's ndcg@5: 0.980355
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005078 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4127
[LightGBM] [Info] Number of data points in the train set: 232674, number of used features: 23
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is

```markdown
# 模型输出结果评价：**非常优秀的水平！**
用大白话+指标解读给你讲，保证你一眼懂这个结果有多好。
```

## 一、输出里的3个关键信息
### 1. 核心指标：NDCG@k（推荐系统排序的「黄金指标」）
- **含义**：NDCG（Normalized Discounted Cumulative Gain）范围是 `0~1`，**越接近1越好**；
- `@1`、`@2`...`@5`：表示给用户推荐Top1、Top2...Top5文章时的排序准确性；
- **你的结果**：
  - NDCG@1：≈0.948（推荐的第1篇文章，94.8%的概率是用户最想点的）；
  - NDCG@5：≈0.980（推荐的前5篇文章，排序几乎完美）。

### 2. 早停情况：`Did not meet early stopping`
- **含义**：训练了100轮（或96、99轮），验证集的NDCG还在稳定，没有出现「连续50轮不提升」的情况；
- **好处**：说明模型**没有过拟合**，还能继续训练（但现在的效果已经很好了）。

### 3. 五折结果的稳定性
- 5折的NDCG@5都在 `0.980~0.981` 之间，波动极小；
- **好处**：说明模型**泛化能力很强**，不是单次运气好，换不同的训练/验证集效果都一样好。

---

## 这个结果意味着什么？
1. **特征工程做得好**：你前面构造的特征（相似度、用户行为、文章属性等）非常有效，能准确预测用户的点击偏好；
2. **模型参数调得好**：LGB排序模型的参数（学习率、树的数量等）设置合理，没有过拟合；
3. **推荐效果有保障**：用这个模型给用户推荐Top5文章，用户大概率会点击排在前面的文章。

---

## 提示（不用太担心）
虽然结果非常好，但可以稍微注意一下：
- 如果你的数据集里「用户历史点击比较丰富」，这个高NDCG是合理的（用户行为多，模型容易学）；
- 如果是「冷启动用户多」的场景，这个结果可能有点偏高，但从五折稳定性看，大概率是真的好。

---

## 总结
**你的LGB排序模型五折交叉验证结果：非常优秀！**

## **LGB分类模型**

## 前置知识：这部分结构属于LGB单模型预测的一部分吗？
**答案：是的！完全属于！**
这部分的结构和前面的「LGB排序模型」**一一对应**，也是分为两部分：
1. **第一部分**：LGB分类模型的**单模型训练**（直接用全部训练数据训练，生成初步推荐结果）；
2. **第二部分**：LGB分类模型的**五折交叉验证**（生成Stacking融合用的特征文件）。

## 二、LGB分类模型 vs 前面的LGB排序模型，核心区别
| 维度 | LGB排序模型 | LGB分类模型 |
|------|------------|------------|
| 模型类 | `lgb.LGBMRanker` | `lgb.LGBMClassifier` |
| 核心思路 | 给用户的文章排顺序（关注相对顺序） | 判断用户“会不会点击”这篇文章（关注绝对概率） |
| 输出 | 相对分数（无固定范围） | 0~1的概率值（点击概率） |
| 评估指标 | NDCG（排序指标） | AUC（分类指标） |
| 特殊要求 | 需要`group`参数（按用户分组） | **不需要**`group`参数 |

## 三、这部分最终产出
和排序模型一样，也是2个文件：
1. `lgb_cls_score.csv`：单模型的预测分数（给加权融合用）；
2. `trn_lgb_cls_feats.csv`、`tst_lgb_cls_feats.csv`：五折交叉验证生成的新特征（给Stacking融合用）。

In [7]:
# ===================== 第一部分：LGB分类模型单模型训练 =====================
# 1. 定义LGB分类模型
# 【注意】：参数和排序模型基本一致，只是模型类换成了LGBMClassifier
#  n_estimators=500（比排序模型的 100 多，因为分类模型可以训练更多轮）。
lgb_Classfication = lgb.LGBMClassifier(boosting_type='gbdt', num_leaves=31, reg_alpha=0.0, reg_lambda=1,
                            max_depth=-1, n_estimators=500, subsample=0.7, colsample_bytree=0.7, subsample_freq=1,
                            learning_rate=0.01, min_child_weight=50, random_state=2018, n_jobs=16, verbose=-1)

# 2. 训练模型
# 【核心区别】：不需要group参数！
#  核心区别 2：评估指标从 ndcg 换成了 auc（分类模型的经典指标，范围 0~1，越接近 1 越好）
if offline:
    # 【版本兼容提示】：如果报错early_stopping_rounds，换成callbacks=[lgb.early_stopping(50)]
    lgb_Classfication.fit(trn_user_item_feats_df_rank_model[lgb_cols], trn_user_item_feats_df_rank_model['label'],
                    eval_set=[(val_user_item_feats_df_rank_model[lgb_cols], val_user_item_feats_df_rank_model['label'])],
                    eval_metric=['auc'], callbacks=[lgb.early_stopping(50)])
else:
    lgb_Classfication.fit(trn_user_item_feats_df_rank_model[lgb_cols], trn_user_item_feats_df_rank_model['label'])

# 3. 模型预测：取正类（点击）的概率
# 【核心区别】：用predict_proba()[:,1]，输出0~1的点击概率    ([:,0] 是不点击的概率，[:,1] 是点击的概率)
tst_user_item_feats_df['pred_score'] = lgb_Classfication.predict_proba(tst_user_item_feats_df[lgb_cols])[:,1]

# 4. 保存单模型预测分数（给加权融合用）
tst_user_item_feats_df[['user_id', 'click_article_id', 'pred_score']].to_csv(save_path + 'lgb_cls_score.csv', index=False)

# 5. 生成单模型提交结果
rank_results = tst_user_item_feats_df[['user_id', 'click_article_id', 'pred_score']]
rank_results['click_article_id'] = rank_results['click_article_id'].astype(int)
submit(rank_results, topk=5, model_name='lgb_cls')

In [8]:
# ===================== 第二部分：LGB分类模型五折交叉验证 =====================
# 1. 定义按用户划分五折的函数（和前面排序模型完全一样）
def get_kfold_users(trn_df, n=5):
    user_ids = trn_df['user_id'].unique()
    user_set = [user_ids[i::n] for i in range(n)]
    return user_set

k_fold = 5
trn_df = trn_user_item_feats_df_rank_model
user_set = get_kfold_users(trn_df, n=k_fold)

score_list = []
score_df = trn_df[['user_id', 'click_article_id', 'label']]
sub_preds = np.zeros(tst_user_item_feats_df_rank_model.shape[0])

# 2. 五折交叉验证循环
for n_fold, valid_user in enumerate(user_set):
    # 2.1 划分训练集和验证集（和排序模型一样）
    train_idx = trn_df[~trn_df['user_id'].isin(valid_user)]
    valid_idx = trn_df[trn_df['user_id'].isin(valid_user)]
    
    # 2.2 定义LGB分类模型（单模型n_estimators=500，这里是100，注意区别）
    lgb_Classfication = lgb.LGBMClassifier(boosting_type='gbdt', num_leaves=31, reg_alpha=0.0, reg_lambda=1,
                            max_depth=-1, n_estimators=100, subsample=0.7, colsample_bytree=0.7, subsample_freq=1,
                            learning_rate=0.01, min_child_weight=50, random_state=2018, n_jobs=16, verbose=-1)
    
    # 2.3 训练模型（不需要group参数，评估指标是AUC）
    # 【版本兼容提示】：如果报错early_stopping_rounds，换成callbacks=[lgb.early_stopping(50)]
    lgb_Classfication.fit(train_idx[lgb_cols], train_idx['label'], eval_set=[(valid_idx[lgb_cols], valid_idx['label'])],
                          eval_metric=['auc'], callbacks=[lgb.early_stopping(50)])
    
    # 2.4 预测验证集结果：取点击概率
    valid_idx['pred_score'] = lgb_Classfication.predict_proba(valid_idx[lgb_cols],
                                                              num_iteration=lgb_Classfication.best_iteration_)[:,1]
    
    # 2.5 重点：分类模型predict_proba()[:,1] 输出已经是 0~1 的概率值，所以验证集这里不需要归一化
    # 但后面测试集还是做了归一化，是为了和其他模型（比如排序模型）融合时，量级保持一致
    # valid_idx['pred_score'] = valid_idx[['pred_score']].transform(lambda x: norm_sim(x))
    
    # 2.6 给验证集的文章按用户排名
    valid_idx.sort_values(by=['user_id', 'pred_score'])
    valid_idx['pred_rank'] = valid_idx.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')
    
    # 2.7 保存验证集预测结果
    score_list.append(valid_idx[['user_id', 'click_article_id', 'pred_score', 'pred_rank']])
    
    # 2.8 预测测试集结果（累加5次）
    if not offline:
        sub_preds += lgb_Classfication.predict_proba(tst_user_item_feats_df_rank_model[lgb_cols],
                                                     num_iteration=lgb_Classfication.best_iteration_)[:,1]

# 3. 拼接训练集的新特征并保存
score_df_ = pd.concat(score_list, axis=0)
score_df = score_df.merge(score_df_, how='left', on=['user_id', 'click_article_id'])
score_df[['user_id', 'click_article_id', 'pred_score', 'pred_rank', 'label']].to_csv(save_path + 'trn_lgb_cls_feats.csv', index=False)

# 4. 平均测试集的预测结果，生成测试集的新特征并保存
tst_user_item_feats_df_rank_model['pred_score'] = sub_preds / k_fold
# 【注意】：测试集这里还是做了归一化，为了和其他模型融合时量级一致
tst_user_item_feats_df_rank_model['pred_score'] = tst_user_item_feats_df_rank_model['pred_score'].transform(lambda x: norm_sim(x))
tst_user_item_feats_df_rank_model.sort_values(by=['user_id', 'pred_score'])
tst_user_item_feats_df_rank_model['pred_rank'] = tst_user_item_feats_df_rank_model.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')
tst_user_item_feats_df_rank_model[['user_id', 'click_article_id', 'pred_score', 'pred_rank']].to_csv(save_path + 'tst_lgb_cls_feats.csv', index=False)

# 5. 生成五折交叉验证后的单模型提交结果
rank_results = tst_user_item_feats_df_rank_model[['user_id', 'click_article_id', 'pred_score']]
rank_results['click_article_id'] = rank_results['click_article_id'].astype(int)
submit(rank_results, topk=5, model_name='lgb_cls')

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.818278	valid_0's binary_logloss: 0.443321
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.822842	valid_0's binary_logloss: 0.445084
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.816607	valid_0's binary_logloss: 0.445907
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.820297	valid_0's binary_logloss: 0.446919
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.819724	valid_0's binary_logloss: 0.446732


LGB 分类模型训练效果非常好！可放心进入模型融合！

## **DIN模型**

# 前置知识：DIN模型是什么 + 这部分结构
## 一、DIN模型是什么？和前面LGB的核心区别
DIN（Deep Interest Network，深度兴趣网络）是**阿里2018年提出的深度学习推荐模型**，核心优势是：
- **LGB模型**：把所有特征“平等”看待，无法特别突出“用户历史点击里和当前候选文章相关的部分”；
- **DIN模型**：引入**注意力机制（Attention）**，给用户历史点击里“和当前候选文章像的文章”更高的权重，动态捕捉用户兴趣。

## 二、这部分代码的结构
和前面LGB模型**完全一一对应**：
1. **第一部分**：构造用户历史行为序列 + DIN单模型训练；
2. **第二部分**：DIN五折交叉验证（生成Stacking融合用的特征）。

## 三、这部分的3个关键注意事项
 **GPU设置**：代码里有 `os.environ["CUDA_VISIBLE_DEVICES"] = "2"`，如果你没有GPU，要注释掉这行；       
 **归一化**：神经网络**必须**对连续特征归一化，这是和LGB模型的重要区别（LGB对归一化不敏感）。

In [9]:
# %pip uninstall -y tensorflow keras deepctr
# 使用清华镜像源强行安装，增加超时时间，确保一次成功
# 使用阿里云镜像，并强制信任该主机（绕过 SSL 报错）
# %pip install tensorflow==2.15.0 deepctr -i http://mirrors.aliyun.com/pypi/simple/ --trusted-host mirrors.aliyun.com --default-timeout=1000

In [10]:
# %pip install keras==2.15.0

In [11]:
# ===================== 第一部分：构造用户历史行为序列 + DIN单模型训练 =====================
# 1. 构造用户历史点击行为序列（DIN模型的核心输入）
if offline:
    all_data = pd.read_csv('./data/train_click_log.csv')
else:
    trn_data = pd.read_csv('./data/train_click_log.csv')
    tst_data = pd.read_csv('./data/testA_click_log.csv')
    # 【修改点1】：pandas新版本弃用append，改成pd.concat
    # all_data = trn_data.append(tst_data)
    all_data = pd.concat([trn_data, tst_data], axis=0, ignore_index=True)

# 按用户分组，把每个用户点击过的文章ID聚合成一个列表
# 构造用户历史行为序列（DIN 特有） 如 [123, 456, 789]
# LGB 模型用的是 “统计特征”（比如历史点击次数、平均相似度），而 DIN 用的是 “原始序列”，能保留更多信息
hist_click = all_data[['user_id', 'click_article_id']].groupby('user_id').agg(list).reset_index()
his_behavior_df = pd.DataFrame()
his_behavior_df['user_id'] = hist_click['user_id']
his_behavior_df['hist_click_article_id'] = hist_click['click_article_id']

# 2. 数据备份（和前面LGB一样）
trn_user_item_feats_df_din_model = trn_user_item_feats_df.copy()

if offline:
    val_user_item_feats_df_din_model = val_user_item_feats_df.copy()
else:
    val_user_item_feats_df_din_model = None

tst_user_item_feats_df_din_model = tst_user_item_feats_df.copy()

# 3. 把历史行为序列合并到特征数据里
trn_user_item_feats_df_din_model = trn_user_item_feats_df_din_model.merge(his_behavior_df, on='user_id')

if offline:
    val_user_item_feats_df_din_model = val_user_item_feats_df_din_model.merge(his_behavior_df, on='user_id')
else:
    val_user_item_feats_df_din_model = None

tst_user_item_feats_df_din_model = tst_user_item_feats_df_din_model.merge(his_behavior_df, on='user_id')

# 4. 导入deepctr和tensorflow相关库

import os
# 1. 强制使用 Keras 2 (针对可能存在的 Keras 3 冲突)
# os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
# tf.compat.v1.disable_eager_execution()
# 3. 屏蔽冗余日志
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
# 接下来再写你原本的 import 语句
from deepctr.models import DIN
from deepctr.feature_column import SparseFeat, VarLenSparseFeat, DenseFeat, get_feature_names
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K
from tensorflow.keras.layers import *
from tensorflow.keras.models import *
from tensorflow.keras.callbacks import *

# 【修改点2】：如果没有GPU，注释掉这两行；如果有GPU，改成你想用的GPU编号（比如"0"）
# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"

max_article_id = all_data['click_article_id'].max() + 1
# 5. 定义DIN数据准备函数
# 5. 定义DIN数据准备函数 (修改版)
def get_din_feats_columns(df, dense_fea, sparse_fea, behavior_fea, his_behavior_fea, emb_dim=32, max_len=100):
    # 5.1 离散型特征：不再用 nunique，统一改用全量数据的 max() + 1
    sparse_feature_columns = []
    for feat in sparse_fea:
        # 获取该特征在 trn 和 tst 中的全局最大值，确保不越界
        # 加上 trn_user_item_feats_df_din_model 和 tst_user_item_feats_df_din_model 的全局判断
        max_val = max(trn_user_item_feats_df_din_model[feat].max(), 
                      tst_user_item_feats_df_din_model[feat].max())
        
        vocab_size = int(max_val) + 2 # +2 是最保险的，留出足够的空位
        sparse_feature_columns.append(SparseFeat(feat, vocabulary_size=vocab_size, embedding_dim=emb_dim))

    # 5.2 连续型特征：保持不变
    dense_feature_columns = [DenseFeat(feat, 1, ) for feat in dense_fea]

    # 5.3 历史行为序列特征：文章 ID 同样使用全局最大值
    # 这里我们提前算好 click_article_id 的全局最大值
    global_max_article_id = int(max(trn_user_item_feats_df_din_model['click_article_id'].max(), 
                                    tst_user_item_feats_df_din_model['click_article_id'].max())) + 2

    var_feature_columns = [VarLenSparseFeat(SparseFeat(feat, 
                                                       vocabulary_size=global_max_article_id, 
                                                       embedding_dim=emb_dim, 
                                                       embedding_name='click_article_id'), 
                                            maxlen=max_len) for feat in his_behavior_fea]
    
    # 5.4 合并所有特征列
    dnn_feature_columns = sparse_feature_columns + dense_feature_columns + var_feature_columns

    # 5.5 构造模型输入字典x
    x = {}
    for name in get_feature_names(dnn_feature_columns):
        if name in his_behavior_fea:
            his_list = [l for l in df[name]]
            x[name] = pad_sequences(his_list, maxlen=max_len, padding='post')
        else:
            x[name] = df[name].values
    return x, dnn_feature_columns

# 6. 把特征分成三类：离散型、连续型、历史行为型
sparse_fea = ['user_id', 'click_article_id', 'category_id', 'click_environment', 'click_deviceGroup',
              'click_os', 'click_country', 'click_region', 'click_referrer_type', 'is_cat_hab']
behavior_fea = ['click_article_id']  # 候选文章ID（和历史行为对应的特征）
hist_behavior_fea = ['hist_click_article_id']  # 用户历史点击文章ID序列
dense_fea = ['sim0', 'time_diff0', 'word_diff0', 'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score',
             'rank', 'click_size', 'time_diff_mean', 'active_level', 'user_time_hob1', 'user_time_hob2',
             'words_hbo', 'words_count']

# 7. 【核心】连续特征归一化（神经网络必须做！LGB对归一化不敏感，但神经网络非常敏感）
mm = MinMaxScaler()

# 【可选】：如果有inf值，取消下面两行的注释
# trn_user_item_feats_df_din_model.replace([np.inf, -np.inf], 0, inplace=True)
# tst_user_item_feats_df_din_model.replace([np.inf, -np.inf], 0, inplace=True)

for feat in dense_fea:
    trn_user_item_feats_df_din_model[feat] = mm.fit_transform(trn_user_item_feats_df_din_model[[feat]])
    if val_user_item_feats_df_din_model is not None:
        val_user_item_feats_df_din_model[feat] = mm.fit_transform(val_user_item_feats_df_din_model[[feat]])
    tst_user_item_feats_df_din_model[feat] = mm.fit_transform(tst_user_item_feats_df_din_model[[feat]])

# 8. 准备训练数据
x_trn, dnn_feature_columns = get_din_feats_columns(trn_user_item_feats_df_din_model, dense_fea,
                                               sparse_fea, behavior_fea, hist_behavior_fea, max_len=50)
y_trn = trn_user_item_feats_df_din_model['label'].values

if offline:
    # 准备验证数据
    x_val, dnn_feature_columns = get_din_feats_columns(val_user_item_feats_df_din_model, dense_fea,
                                                   sparse_fea, behavior_fea, hist_behavior_fea, max_len=50)
    y_val = val_user_item_feats_df_din_model['label'].values

# 9. 准备测试数据（先把label从dense_fea里去掉）
dense_fea = [x for x in dense_fea if x != 'label']
x_tst, dnn_feature_columns = get_din_feats_columns(tst_user_item_feats_df_din_model, dense_fea,
                                               sparse_fea, behavior_fea, hist_behavior_fea, max_len=50)

# 10. 建立DIN模型
model = DIN(dnn_feature_columns, behavior_fea)

# 查看模型结构（可选，看一下参数量）
model.summary()

# 11. 模型编译：优化器adam，损失函数binary_crossentropy，评估指标AUC
model.compile('adam', 'binary_crossentropy', metrics=['binary_crossentropy', tf.keras.metrics.AUC()])

# 12. 模型训练
if offline:
    history = model.fit(x_trn, y_trn, verbose=1, epochs=10, validation_data=(x_val, y_val), batch_size=256)
else:
    # 也可以用validation_split=0.3从训练集里分验证集
    # history = model.fit(x_trn, y_trn, verbose=1, epochs=3, validation_split=0.3, batch_size=256)
    history = model.fit(x_trn, y_trn, verbose=1, epochs=2, batch_size=256)

# 13. 模型预测
tst_user_item_feats_df_din_model['pred_score'] = model.predict(x_tst, verbose=1, batch_size=256)

# 14. 保存单模型预测分数（给加权融合用）
tst_user_item_feats_df_din_model[['user_id', 'click_article_id', 'pred_score']].to_csv(save_path + 'din_rank_score.csv', index=False)

# 15. 生成单模型提交结果
rank_results = tst_user_item_feats_df_din_model[['user_id', 'click_article_id', 'pred_score']]
rank_results['click_article_id'] = rank_results['click_article_id'].astype(int)
submit(rank_results, topk=5, model_name='din')



Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 user_id (InputLayer)        [(None, 1)]                  0         []                            
                                                                                                  
 click_article_id (InputLay  [(None, 1)]                  0         []                            
 er)                                                                                              
                                                                                                  
 category_id (InputLayer)    [(None, 1)]                  0         []                            
                                                                                                  
 click_environment (InputLa  [(None, 1)]                  0         []                      

# ===================== 第二部分：DIN五折交叉验证 =====================
# 1. 定义按用户划分五折的函数（和前面完全一样）
def get_kfold_users(trn_df, n=5):
    user_ids = trn_df['user_id'].unique()
    user_set = [user_ids[i::n] for i in range(n)]
    return user_set

k_fold = 5
trn_df = trn_user_item_feats_df_din_model
user_set = get_kfold_users(trn_df, n=k_fold)

score_list = []
score_df = trn_df[['user_id', 'click_article_id', 'label']]
sub_preds = np.zeros(tst_user_item_feats_df_rank_model.shape[0])

# 2. 准备测试数据（和前面单模型一样）
dense_fea = [x for x in dense_fea if x != 'label']
x_tst, dnn_feature_columns = get_din_feats_columns(tst_user_item_feats_df_din_model, dense_fea,
                                                   sparse_fea, behavior_fea, hist_behavior_fea, max_len=50)

# 3. 五折交叉验证循环
for n_fold, valid_user in enumerate(user_set):
    train_idx = trn_df[~trn_df['user_id'].isin(valid_user)]
    valid_idx = trn_df[trn_df['user_id'].isin(valid_user)]

    # 3.1 准备训练数据
    x_trn, dnn_feature_columns = get_din_feats_columns(train_idx, dense_fea,
                                                       sparse_fea, behavior_fea, hist_behavior_fea, max_len=50)
    y_trn = train_idx['label'].values

    # 3.2 准备验证数据
    x_val, dnn_feature_columns = get_din_feats_columns(valid_idx, dense_fea,
                                                   sparse_fea, behavior_fea, hist_behavior_fea, max_len=50)
    y_val = valid_idx['label'].values

    # 3.3 【注意】：这里要重新建立模型！不能用前面的单模型
    model = DIN(dnn_feature_columns, behavior_fea)
    model.compile('adam', 'binary_crossentropy', metrics=['binary_crossentropy', tf.keras.metrics.AUC()])

    # 3.4 训练模型
    history = model.fit(x_trn, y_trn, verbose=1, epochs=2, validation_data=(x_val, y_val), batch_size=256)

    # 3.5 预测验证集结果
    valid_idx['pred_score'] = model.predict(x_val, verbose=1, batch_size=256)

    valid_idx.sort_values(by=['user_id', 'pred_score'])
    valid_idx['pred_rank'] = valid_idx.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')

    # 3.6 保存验证集预测结果
    score_list.append(valid_idx[['user_id', 'click_article_id', 'pred_score', 'pred_rank']])

    # 3.7 预测测试集结果（累加5次）
    if not offline:
        # 【修改点3】：DIN的predict输出是二维数组(n,1)，要[:,0]取出来变成一维
        sub_preds += model.predict(x_tst, verbose=1, batch_size=256)[:, 0]

# 4. 拼接训练集的新特征并保存
score_df_ = pd.concat(score_list, axis=0)
score_df = score_df.merge(score_df_, how='left', on=['user_id', 'click_article_id'])
score_df[['user_id', 'click_article_id', 'pred_score', 'pred_rank', 'label']].to_csv(save_path + 'trn_din_cls_feats.csv', index=False)

# 5. 平均测试集的预测结果，生成测试集的新特征并保存
tst_user_item_feats_df_din_model['pred_score'] = sub_preds / k_fold
tst_user_item_feats_df_din_model['pred_score'] = tst_user_item_feats_df_din_model['pred_score'].transform(lambda x: norm_sim(x))
tst_user_item_feats_df_din_model.sort_values(by=['user_id', 'pred_score'])
tst_user_item_feats_df_din_model['pred_rank'] = tst_user_item_feats_df_din_model.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')
tst_user_item_feats_df_din_model[['user_id', 'click_article_id', 'pred_score', 'pred_rank']].to_csv(save_path + 'tst_din_cls_feats.csv', index=False)

## **加权融合**

模型融合（加权融合 + Stacking 融合）             
核心前提            
加权融合：完全不用五折结果！只需要 3 个模型的单模型预测文件（你已经全部跑完，无任何问题）         
Stacking 融合：原版需要 3 个模型的五折特征文件，但你没跑 DIN 五折 → 必须删除 DIN 相关代码，否则会报错找不到文件            

# 读取多个模型的排序结果文件（单模型预测结果，和五折无关）
lgb_ranker = pd.read_csv(save_path + 'lgb_ranker_score.csv')   # LGB排序模型单模型结果
lgb_cls = pd.read_csv(save_path + 'lgb_cls_score.csv')         # LGB分类模型单模型结果
din_ranker = pd.read_csv(save_path + 'din_rank_score.csv')     # DIN模型单模型结果

# 把三个模型装进字典，方便调用
rank_model = {'lgb_ranker': lgb_ranker, 'lgb_cls': lgb_cls, 'din_ranker': din_ranker}

# 定义加权融合函数
def get_ensumble_predict_topk(rank_model, topk=5):
    # 第一步：合并 LGB分类 + DIN 的结果
    final_recall = rank_model['lgb_cls'].append(rank_model['din_ranker'])
    # 第二步：把 LGB排序的分数归一化（统一三个模型的分数范围）
    rank_model['lgb_ranker']['pred_score'] = rank_model['lgb_ranker']['pred_score'].transform(lambda x: norm_sim(x))
    # 第三步：把归一化后的 LGB排序 也合并进来
    final_recall = final_recall.append(rank_model['lgb_ranker'])
    # 核心逻辑：同一个用户+同一篇文章，把三个模型的预测分数 相加
    final_recall = final_recall.groupby(['user_id', 'click_article_id'])['pred_score'].sum().reset_index()
    # 生成最终提交文件
    submit(final_recall, topk=topk, model_name='ensemble_fuse')

# 执行融合
get_ensumble_predict_topk(rank_model)

In [12]:
# ===================== 加权融合（修复版，对标原版逻辑） =====================
# 读取三个单模型预测结果（你已全部生成，直接用）
lgb_ranker = pd.read_csv(save_path + 'lgb_ranker_score.csv')
lgb_cls = pd.read_csv(save_path + 'lgb_cls_score.csv')
din_ranker = pd.read_csv(save_path + 'din_rank_score.csv')

rank_model = {'lgb_ranker': lgb_ranker, 'lgb_cls': lgb_cls, 'din_ranker': din_ranker}

def get_ensumble_predict_topk(rank_model, topk=5):
    # 【修改1】原代码：final_recall = rank_model['lgb_cls'].append(rank_model['din_ranker'])
    # 【修改原因】pandas弃用append，替换为concat
    final_recall = pd.concat([rank_model['lgb_cls'], rank_model['din_ranker']], axis=0, ignore_index=True)
    
    rank_model['lgb_ranker']['pred_score'] = rank_model['lgb_ranker']['pred_score'].transform(lambda x: norm_sim(x))
    
    # 【修改2】原代码：final_recall = final_recall.append(rank_model['lgb_ranker'])
    final_recall = pd.concat([final_recall, rank_model['lgb_ranker']], axis=0, ignore_index=True)
    
    final_recall = final_recall.groupby(['user_id', 'click_article_id'])['pred_score'].sum().reset_index()
    submit(final_recall, topk=topk, model_name='ensemble_fuse')

get_ensumble_predict_topk(rank_model)

In [ ]:
# stacking 融合 原版
# 读取三个模型的五折交叉验证特征（你没跑DIN五折，这里会报错！）
trn_lgb_ranker_feats = pd.read_csv(save_path + 'trn_lgb_ranker_feats.csv')   # 有
trn_lgb_cls_feats = pd.read_csv(save_path + 'trn_lgb_cls_feats.csv')       # 有
trn_din_cls_feats = pd.read_csv(save_path + 'trn_din_cls_feats.csv')       # 无 → 报错

tst_lgb_ranker_feats = pd.read_csv(save_path + 'tst_lgb_ranker_feats.csv') # 有
tst_lgb_cls_feats = pd.read_csv(save_path + 'tst_lgb_cls_feats.csv')       # 有
tst_din_cls_feats = pd.read_csv(save_path + 'tst_din_cls_feats.csv')       # 无 → 报错

# 拼接三个模型的特征
finall_trn_ranker_feats = trn_lgb_ranker_feats[['user_id', 'click_article_id', 'label']]
finall_tst_ranker_feats = tst_lgb_ranker_feats[['user_id', 'click_article_id']]

# 循环添加3个模型的 pred_score / pred_rank
for idx, trn_model in enumerate([trn_lgb_ranker_feats, trn_lgb_cls_feats, trn_din_cls_feats]):
    for feat in [ 'pred_score', 'pred_rank']:
        col_name = feat + '_' + str(idx)
        finall_trn_ranker_feats[col_name] = trn_model[feat]

for idx, tst_model in enumerate([tst_lgb_ranker_feats, tst_lgb_cls_feats, tst_din_cls_feats]):
    for feat in [ 'pred_score', 'pred_rank']:
        col_name = feat + '_' + str(idx)
        finall_tst_ranker_feats[col_name] = tst_model[feat]

# 6个特征（3个模型×2）
feat_cols = ['pred_score_0', 'pred_rank_0', 'pred_score_1', 'pred_rank_1', 'pred_score_2', 'pred_rank_2']

# 训练逻辑回归，做第二层融合
lr = LogisticRegression()
lr.fit(trn_x, trn_y)
finall_tst_ranker_feats['pred_score'] = lr.predict_proba(tst_x)[:, 1]

rank_results = finall_tst_ranker_feats[['user_id', 'click_article_id', 'pred_score']]
submit(rank_results, topk=5, model_name='ensumble_staking')

In [13]:
# ===================== Stacking 融合（修改版，适配你未跑DIN五折） =====================
# 读取五折特征：【修改1】原代码读取3个模型，现删除DIN，只保留2个你跑过的模型
# 原代码：trn_din_cls_feats = pd.read_csv(...) + tst_din_cls_feats = pd.read_csv(...)
# 修改原因：你没跑DIN五折，无对应文件
trn_lgb_ranker_feats = pd.read_csv(save_path + 'trn_lgb_ranker_feats.csv')
trn_lgb_cls_feats = pd.read_csv(save_path + 'trn_lgb_cls_feats.csv')

tst_lgb_ranker_feats = pd.read_csv(save_path + 'tst_lgb_ranker_feats.csv')
tst_lgb_cls_feats = pd.read_csv(save_path + 'tst_lgb_cls_feats.csv')

# 拼接基础列（和原版完全一致）
finall_trn_ranker_feats = trn_lgb_ranker_feats[['user_id', 'click_article_id', 'label']]
finall_tst_ranker_feats = tst_lgb_ranker_feats[['user_id', 'click_article_id']]

# 【修改2】原代码循环3个模型，现只循环2个模型（删除trn_din_cls_feats）
for idx, trn_model in enumerate([trn_lgb_ranker_feats, trn_lgb_cls_feats]):
    for feat in ['pred_score', 'pred_rank']:
        col_name = feat + '_' + str(idx)
        finall_trn_ranker_feats[col_name] = trn_model[feat]

# 【修改3】原代码循环3个模型，现只循环2个模型（删除tst_din_cls_feats）
for idx, tst_model in enumerate([tst_lgb_ranker_feats, tst_lgb_cls_feats]):
    for feat in ['pred_score', 'pred_rank']:
        col_name = feat + '_' + str(idx)
        finall_tst_ranker_feats[col_name] = tst_model[feat]

# 【修改4】原特征列6个，现改为4个（2个模型×2个特征）
# 原代码：feat_cols = ['pred_score_0','pred_rank_0','pred_score_1','pred_rank_1','pred_score_2','pred_rank_2']
feat_cols = ['pred_score_0', 'pred_rank_0', 'pred_score_1', 'pred_rank_1']

# 以下代码【完全和原版一致，无任何修改】
trn_x = finall_trn_ranker_feats[feat_cols]
trn_y = finall_trn_ranker_feats['label']
tst_x = finall_tst_ranker_feats[feat_cols]

from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(trn_x, trn_y)

finall_tst_ranker_feats['pred_score'] = lr.predict_proba(tst_x)[:, 1]

rank_results = finall_tst_ranker_feats[['user_id', 'click_article_id', 'pred_score']]
submit(rank_results, topk=5, model_name='ensumble_staking')